<a href="https://colab.research.google.com/github/waelrash1/ai-for-business-analytics/blob/main/notebooks/Block2_Code_Lab_Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelling and evaluation with Python and an AI partner

**MI6261 AI for Business Analytics, Block 2.** Dr Wael Rashwan, School of Business, Maynooth University. MSc in Business Analytics.

Two questions an online shop's operations manager asks, and the two main tasks of predictive analytics:

- **Regression (tasks 2 to 6):** when a customer checks out, what shipping charge (freight) should we quote? The answer is a number.
- **Classification (tasks 7 to 12):** when the seller hands a parcel to the carrier, will it arrive late? The answer is yes or no. Late parcels get most of the bad reviews, so an early warning lets customer care act first.

In each task you:

1. **Plan**: read the question and decide what one result you need.
2. **Prompt**: ask Gemini or Claude for **one small code cell**.
3. **Read** the code before you run it: is anything fitted on the test set? Is a leaking column used?
4. **Run** it.
5. **Verify** one number in Excel or by a second method.
6. **Explain** the result in your own words, for a manager.

The prompt help fades: tasks 1 to 3 give you the prompt and I show them on screen; tasks 4, 5 and 7 give you the prompt; tasks 6 and 8 to 12 you write yourself.

**Rules.** Use only what is known at the moment of the decision. Split before you fit. Beat a baseline before you trust a model. Score on the test set once.

**Where this data comes from.** The same Olist orders, cleaned with exactly the rules of the data cleaning lab:

1. Repeated rows removed, so there is one row per order.
2. Text dates turned into dates, then into day counts (delivery_days, days_late, promised_days, handling_days).
3. Only orders with status delivered **and** a delivery date kept; cancelled and undelivered orders left out.
4. State codes made consistent: two capital letters.
5. Orders with no product weight or no address location left out, because the models need them (fewer than 1 in 100).

It is a larger sample than your cleaned file (5,000 real delivered orders from the Olist Brazilian E-Commerce Public Dataset, Kaggle, CC BY-NC-SA 4.0), with five extra columns joined from the other Olist tables: product weight, the distance from seller to customer (between zip-code areas, so approximate), the seller's state, the promised days and the days the seller took to hand the parcel to the carrier. **Task 1 reruns your cleaning checks on it before any modelling: do not trust a table you have not checked.**

- **Orders**: https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv

**AI use.** You may use any AI to write code. You must read, check and explain every cell. Keep a note of what you had to correct.

---
## Task 1: Load the cleaned orders, check them, add the target

**Question.** Is this table as clean as the one you made, and what will each model predict?

**Plan.** One cell that produces: `n_rows`, `n_late`, `n_dup_ids`, `n_missing`.

**A weak prompt:** *"Load the data and build a model."* Which model, predicting what, on data you have checked? Rerun your cleaning checks first, then name the targets.

In [ ]:
import pandas as pd
orders_url = "https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv"

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: Colab. `orders_url`: 5,000 cleaned, delivered orders with freight, weight_g, distance_km, both states, handling_days, days_late and more.
Task: Load as `orders` (order_id as text). Count repeated ids, missing values, bad state codes, negative values. Add late: 1 if days_late above 0.
Constraints: pandas only; drop or change nothing.
Format: One cell; label each check.
Check: Store `n_rows`, `n_late`, `n_dup_ids` and `n_missing`.
```

**Paste the AI's code below.** Before you run it, check that it:
- The checks only count: nothing is dropped
- Each check is one of your cleaning-lab checks
- late uses > 0: on the promised day is on time

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Verify in Excel.**

- Late orders in olist_model_orders.csv: `=COUNTIF(N2:N5001,">0")`

In [ ]:
excel_value = None   # type what Excel shows
print(n_late == excel_value)   # True means Python and Excel agree

**Explain.** Three checks pass, but 4 orders were handed to the carrier before they were bought: a real recording error in Olist's data. Should they stay in the models? What would you ask the data owner?

In [ ]:
my_explanation = """

"""

---
## Task 2: Regression: features known at checkout, then split

**Question.** When a customer checks out, can we quote the shipping charge?

**Plan.** One cell that produces: `n_train_r`, `mean_train_freight`.

**A weak prompt:** *"Predict freight from all the other columns."* handling_days and delivery_days are only known after the sale. At checkout you know the basket and the addresses, nothing more.

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: `orders`. Target: freight (R$). Known at checkout: weight_g, distance_km, n_items, price, customer_state, seller_state.
Task: X_r (those six) and y_r; split 75/25.
Constraints: train_test_split, random_state=42. No delivery or review columns.
Format: One cell; sizes and mean train freight.
Check: Store `n_train_r` and `mean_train_freight` (2 decimals).
```

**Paste the AI's code below.** Before you run it, check that it:
- Only the six checkout features
- random_state=42
- Nothing fitted yet

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: train and test together are all 5,000 orders

In [ ]:
print(n_train_r + len(Xr_test) == len(orders))   # train and test are all the orders

**Explain.** Why are handling_days and delivery_days not allowed here, even if they helped?

In [ ]:
my_explanation = """

"""

---
## Task 3: The baseline: quote the average

**Question.** How good is one flat shipping charge for everyone?

**Plan.** One cell that produces: `mae_base`.

**A weak prompt:** *"Train the best regression model."* Best compared with what? Without a baseline you cannot tell whether a model has learned anything.

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: `yr_train`, `yr_test`: freight in R$.
Task: Predict the training mean for every test order. Compute the mean absolute error (MAE) on test. Save actual and baseline as reg_test.csv.
Constraints: Use only the training mean. sklearn's mean_absolute_error.
Format: One cell. Print the MAE with a label.
Check: Store `mae_base` (2 decimals).
```

**Paste the AI's code below.** Before you run it, check that it:
- The training mean, not the test mean
- MAE on the test set
- index=False

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Verify in Excel.**

- Open reg_test.csv (download it from the Files pane): 1,250 rows
- Mean absolute error: `=AVERAGE(ABS(A2:A1251-B2:B1251))`

In [ ]:
excel_value = None   # type what Excel shows
print(round(excel_value, 2) == mae_base)   # True means Python and Excel agree

**Explain.** Every model from now on must beat this number. Why use the training mean and not the test mean?

In [ ]:
my_explanation = """

"""

---
## Task 4: Linear regression in a pipeline

**Question.** Does a weighted sum of weight, distance and the rest beat the flat quote?

**Plan.** One cell that produces: `mae_lin`, `r2_lin`.

**A weak prompt:** *"Do a linear regression and give me the accuracy."* Accuracy is for classes, not numbers. For a number, ask for MAE, RMSE and R squared on the test set.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `Xr_train`, `Xr_test`, `yr_train`, `yr_test`. The two states are categories; the rest are numbers.
Task: A Pipeline: scale the numbers, one-hot both states (under 20 orders grouped), then LinearRegression. Fit on train; MAE, RMSE and R squared on test.
Constraints: Fit on train only. handle_unknown='infrequent_if_exist'.
Format: One cell. Print the three measures with labels.
Check: Store `mae_lin` and `r2_lin` (MAE 2 decimals, R squared 3).
```

**Paste the AI's code below.** Before you run it, check that it:
- fit on train; predict on test
- MAE, RMSE and R squared, not accuracy
- Both states encoded, not used as text

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the MAE by hand, from the errors, equals sklearn's

In [ ]:
print(round(np.mean(np.abs(yr_test - pred_lin)), 2) == mae_lin)   # MAE by hand equals sklearn's

**Explain.** The MAE fell by about half. In plain words for a manager, what does R squared tell you here?

In [ ]:
my_explanation = """

"""

---
## Task 5: A random forest, and the overfitting gap

**Question.** Can a more flexible model do better, and is it honest?

**Plan.** One cell that produces: `mae_rf`, `r2_rf`, `train_mae_rf`.

**A weak prompt:** *"Use a more powerful model to get a better score."* Powerful models can memorise the training rows. Always compare the training and test errors.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `prep_r` from task 4; the same train and test data.
Task: A Pipeline with prep_r and RandomForestRegressor. MAE and R squared on test, and MAE on train.
Constraints: n_estimators=200, min_samples_leaf=5, random_state=42.
Format: One cell. Print the measures with labels.
Check: Store `mae_rf`, `r2_rf` and `train_mae_rf`.
```

**Paste the AI's code below.** Before you run it, check that it:
- random_state set, so the result repeats
- Both train and test MAE
- The same prep_r, fitted inside the pipeline

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the training error is lower than the test error: the gap is the overfitting

In [ ]:
print(train_mae_rf < mae_rf)   # the forest fits its training rows better than new ones

**Explain.** The forest beats the linear model on MAE. Is the gain worth a model that is harder to explain to sellers?

In [ ]:
my_explanation = """

"""

---
## Task 6: Look at the errors

**Question.** Where is the quote badly wrong?

**Plan.** One cell that produces: `n_off_10`.

**A weak prompt:** *"Plot the results."* Which results? Ask for predicted against actual, and for the biggest errors: that is where the lessons are.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `yr_test` and `pred_rf` from task 5; `Xr_test`.
- **Task:** Scatter actual against predicted freight with a 45-degree line. Show the 5 orders with the largest absolute error. Count orders off by more than R$10.
- **Constraints:** matplotlib; axes from 0 to 100; label both axes.
- **Format:** One cell: one chart and one small table.
- **Check:** Store the count in `n_off_10`.

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- actual on x, predicted on y, with a diagonal
- Absolute errors for the ranking
- Axis labels and a title that says something

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the mean of the absolute errors is the MAE from task 5

In [ ]:
print(round(errors.abs().mean(), 2) == mae_rf)   # the mean absolute error is the MAE

**Explain.** Look at the five worst orders. What do they have in common, and what would you tell the pricing team?

In [ ]:
my_explanation = """

"""

---
## Task 7: Classification: features known at handover, stratified split

**Question.** When the seller hands a parcel to the carrier, can we tell if it will be late?

**Plan.** One cell that produces: `n_late_test`, `base_acc`.

**A weak prompt:** *"Split the data for classification."* Without stratify, the rare late orders (about 7 in 100) can bunch in train or test, and every score after moves.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `orders`; target late; known at handover: handling_days, promised_days, distance_km, weight_g, freight, n_items, purchase_month, customer_state, seller_state.
Task: X_c and y_c; split 75/25. Accuracy of always predicting 'on time' on test.
Constraints: random_state=42, stratify=y_c; never delivery_days or days_late.
Format: One cell; late share in each part; baseline accuracy.
Check: Store `n_late_test` and `base_acc` (3 decimals).
```

**Paste the AI's code below.** Before you run it, check that it:
- stratify=y_c
- No delivery_days, days_late or review_score in X_c
- The baseline uses the test set only to score

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: train and test have almost the same late share: that is what stratify does

In [ ]:
print(abs(yc_train.mean() - yc_test.mean()) < 0.01)   # stratify keeps the bad share equal

**Explain.** The baseline is right 93 times in 100 and warns nobody. What does that tell you about accuracy here?

In [ ]:
my_explanation = """

"""

---
## Task 8: A business rule as a second baseline

**Question.** How well does 'the seller was slow to ship' flag late parcels?

**Plan.** One cell that produces: `rule_precision`, `rule_recall`.

**A weak prompt:** *"Is slow handling a good predictor?"* Good by what measure? Ask for precision and recall of a simple rule, so a model has something real to beat.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `Xc_test` has handling_days; `yc_test` is late.
- **Task:** Rule: flag when handling_days is 5 or more. Show the confusion matrix; compute precision and recall.
- **Constraints:** sklearn's confusion_matrix, precision_score, recall_score; no model.
- **Format:** One cell. The matrix, then two labelled numbers.
- **Check:** Store `rule_precision` and `rule_recall` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The rule, not a model
- Rows are actual, columns predicted
- Precision and recall, not only accuracy

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: precision by hand: true positives / all flagged

In [ ]:
print(round(((rule == 1) & (yc_test == 1)).sum() / rule.sum(), 3) == rule_precision)   # precision by hand

**Explain.** In words a customer care manager would use, what do this precision and this recall mean?

In [ ]:
my_explanation = """

"""

---
## Task 9: Logistic regression and the confusion matrix

**Question.** Does a model beat the rule?

**Plan.** One cell that produces: `tp_20`, `recall_20`.

**A weak prompt:** *"Build a logistic regression and show the accuracy."* Accuracy hides the misses. Ask for the confusion matrix, precision and recall, at a stated threshold.

**Write your prompt.** Use the five headings. Hints:

- **Context:** The classification train and test sets; the two states are categories.
- **Task:** Pipeline: scale, one-hot both states, LogisticRegression. Flag where the probability is 0.2 or more; confusion matrix, precision, recall. Save actual and probability as clf_test.csv.
- **Constraints:** Fit on train; max_iter=1000.
- **Format:** One cell.
- **Check:** Store `tp_20` (true positives) and `recall_20` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- predict_proba, column 1
- The threshold is written down: 0.2
- Fitted on train, scored on test

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Verify in Excel.**

- Open clf_test.csv: true positives at 0.2
- True positives: `=COUNTIFS(A2:A1251,1,B2:B1251,">=0.2")`

In [ ]:
excel_value = None   # type what Excel shows
print(tp_20 == excel_value)   # True means Python and Excel agree

**Explain.** Why does 0.5 make little sense when only 7 orders in 100 are late? Compare with the rule from task 8.

In [ ]:
my_explanation = """

"""

---
## Task 10: Move the threshold

**Question.** What if we warn more customers?

**Plan.** One cell that produces: `precision_10`, `recall_10`.

**A weak prompt:** *"Find the best threshold."* Best for whom? A lower threshold catches more late parcels and warns more customers whose parcel was fine.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_late` and `yc_test` from task 9.
- **Task:** For thresholds 0.5, 0.3, 0.2 and 0.1: flagged orders, precision and recall, in one table.
- **Constraints:** No refitting; the same probabilities.
- **Format:** One cell, one small table.
- **Check:** Store precision and recall at 0.1 in `precision_10` and `recall_10` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The same p_late for every threshold
- Flagged count beside precision and recall
- No model is refitted

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: a lower threshold never catches fewer late parcels

In [ ]:
print(recall_10 >= recall_20)   # a lower threshold never catches fewer late parcels

**Explain.** Customer care can send 100 warnings from these 1,250 parcels. Which threshold fits, and what recall do you get?

In [ ]:
my_explanation = """

"""

---
## Task 11: Compare models by AUC

**Question.** Which model ranks late parcels better, at every threshold?

**Plan.** One cell that produces: `auc_log`, `auc_rf`.

**A weak prompt:** *"Which model is better?"* Better at which threshold? AUC compares how well each model ranks orders, before any threshold is chosen.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_late` from the logistic regression; `prep_c`; the classification train and test sets.
- **Task:** Fit a RandomForestClassifier in a pipeline. ROC AUC of both models on test; plot both ROC curves.
- **Constraints:** n_estimators=300, min_samples_leaf=10, random_state=42.
- **Format:** One cell: one chart, two labelled numbers.
- **Check:** Store `auc_log` and `auc_rf` (3 decimals).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- Probabilities, not 0 or 1 predictions, go into the AUC
- Both models on the same test set
- random_state on the forest

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: AUC by hand: the chance a random late parcel scores above a random on-time one

In [ ]:
print(round((p_late[yc_test.values == 1][:, None] > p_late[yc_test.values == 0]).mean() + 0.5 * (p_late[yc_test.values == 1][:, None] == p_late[yc_test.values == 0]).mean(), 3) == auc_log)   # AUC by hand

**Explain.** AUC 0.5 is a coin toss and 1.0 is perfect. Where do these models sit, and is that good enough to act on?

In [ ]:
my_explanation = """

"""

---
## Task 12: Choose the threshold from costs

**Question.** Which threshold costs the business least?

**Plan.** One cell that produces: `best_t`, `best_cost`.

**A weak prompt:** *"What threshold should we use?"* Only costs answer that. Say what a missed late parcel costs and what an unnecessary warning costs.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `p_rf`, `yc_test`. A late parcel nobody warned about costs 5 (a likely bad review); a warning for a parcel that arrives on time costs 1.
- **Task:** For thresholds from 0.05 to 0.95 in steps of 0.05, total cost = 5 x misses + 1 x false alarms. Plot cost against threshold; find the cheapest.
- **Constraints:** No refitting. Round thresholds to 2 decimals.
- **Format:** One cell: one chart, one labelled result.
- **Check:** Store `best_t` and `best_cost`.

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- 5 for a miss, 1 for a false alarm, as asked
- The same p_rf; no refitting
- The chart marks the cheapest threshold

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the best plan costs less than warning nobody

In [ ]:
print(best_cost < 5 * int(yc_test.sum()))   # cheaper than warning nobody

**Explain.** Theory says the best threshold is 1 / (1 + 5), about 0.17. Is yours close? Why might it differ on 1,250 test orders?

In [ ]:
my_explanation = """

"""

---
## Your results table

Fill in one row per model. This table goes in your report.

| Task | Model | Measure on the test set | Value | Beats the baseline? |
|---|---|---|---|---|
| 3 | Average (baseline) | MAE, R$ | | |
| 4 | Linear regression | MAE, R$; R squared | | |
| 5 | Random forest | MAE, R$; R squared | | |
| 8 | Slow-handover rule (baseline) | precision, recall | | |
| 9 | Logistic regression, 0.2 | precision, recall | | |
| 11 | Logistic and random forest | AUC | | |

## Your report (about 300 words)

**Questions.** The two predictions, and the moment each is made.

**Data and features.** Which features, and why each is known at that moment.

**Regression.** Baseline against models, in R$. What the errors show.

**Classification.** Baseline and rule against the models; the threshold you chose and why.

**Limits.** What these models cannot do, and what data would help.

**AI use.** Which AI you used, one prompt that worked well, and one thing in the AI's code you had to correct.